# European Flight Delay Prediction — Pipeline

This notebook implements and tests an Apache Beam data pipeline for the European Flight Delay Prediction project. It reads Parquet flight data generated by '03_data_splitting.ipynb', validates records separating rejected records.

The pipeline will subsequently be extended to support automated preprocessing and reproducible machine-learning datasets.


## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [ ]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"

%cd /content/
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR} will pull latest version")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")
    # Change the current working directory to the project root
    %cd {REPO_DIR}



## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [ ]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME
*   SPLIT_RUN_ID - this should reference existing notebook 03 split run


In [ ]:

from google.colab import auth
auth.authenticate_user()

from pathlib import Path
from google.cloud import storage
import pandas as pd
import json

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"

# Existing Notebook 03 split run
SPLIT_RUN_ID = "20261020T065554Z-cc2d29384141"
SPLIT_PREFIX = f"processed/splits/runs/{SPLIT_RUN_ID}"

TARGET_COLUMN = "delayed_15min"

WORK_DIR = Path("/content/flight_preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TRAIN = WORK_DIR / "flights_train.parquet"
LOCAL_DEV = WORK_DIR / "flights_dev.parquet"
LOCAL_TEST = WORK_DIR / "flights_test.parquet"
LOCAL_SAMPLE = WORK_DIR / "flights_sample_100.parquet"

OUTPUT_DIR = WORK_DIR / "beam_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PREFIX = str(OUTPUT_DIR / "flights_sample")

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# Confirm the source run exists
provenance_blob = bucket.blob(
    f"{SPLIT_PREFIX}/provenance.json"
)

if not provenance_blob.exists():
    raise FileNotFoundError(
        f"Split run not found: gs://{BUCKET_NAME}/{SPLIT_PREFIX}"
    )

source_provenance = json.loads(
    provenance_blob.download_as_text()
)

assert source_provenance["run_id"] == SPLIT_RUN_ID

print("Connected to Google Cloud Storage.")
print("Source split run:", SPLIT_RUN_ID)
print("Source split sizes:", source_provenance["split_rows"])


## 3. Import the Pipeline


In [ ]:
import sys
import inspect
import os

# Allow imports within the notebook
sys.path.insert(0, os.path.abspath("pipeline"))

# Allow imports in separate Python processes (pytest)
os.environ["PYTHONPATH"] = os.path.abspath("pipeline")

import flight_pipeline as fp

## 4. Run Automated Tests with sample data


In [ ]:
!python -m pytest pipeline/tests -v

## 5. Run Pipeline on Sample Flight data

### Download a sample of 100 records from training data

In [ ]:
# Download the training split created by Notebook 03
SOURCE_OBJECT = f"{SPLIT_PREFIX}/flights_train.parquet"

blob = bucket.blob(SOURCE_OBJECT)

if not blob.exists():
    raise FileNotFoundError(
        f"Training file not found: gs://{BUCKET_NAME}/{SOURCE_OBJECT}"
    )

blob.download_to_filename(str(LOCAL_TRAIN))

# Read the first 100 flights for our initial test
sample_df = pd.read_parquet(LOCAL_TRAIN).head(100)

# Save as a separate Parquet file
sample_df.to_parquet(LOCAL_SAMPLE, index=False)

print("Sample records:", len(sample_df))
print("Sample file:", LOCAL_SAMPLE)

display(sample_df.head())

### Run the Apache Beam pipeline

In [ ]:
import apache_beam as beam

with beam.Pipeline(
    runner="DirectRunner"
) as pipeline:

    fp.build(
        pipeline,
        input_pattern=str(LOCAL_SAMPLE),
        output_prefix=OUTPUT_PREFIX
    )

print("Apache Beam pipeline execution completed.")

## 6. Inspect Accepted and Rejected Records

In [ ]:
from pathlib import Path

good_files = sorted(OUTPUT_DIR.glob("flights_sample_good*.jsonl"))
bad_files = sorted(OUTPUT_DIR.glob("flights_sample_bad*.jsonl"))

def read_jsonl_files(files):
    records = []

    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

    return pd.DataFrame(records)

good_df = read_jsonl_files(good_files)
bad_df = read_jsonl_files(bad_files)

print("Accepted records:", len(good_df))
print("Rejected records:", len(bad_df))

display(good_df.head())

## 7. Reconcile Input and Output Counts

In [ ]:
input_count = len(sample_df)
good_count = len(good_df)
bad_count = len(bad_df)

print("Input records:   ", input_count)
print("Accepted records:", good_count)
print("Rejected records:", bad_count)

assert input_count == good_count + bad_count, (
    "Reconciliation failed: input and output counts differ"
)

print("Reconciliation passed.")

## 8. Create new dataset containing Invalid Records

In [ ]:
# Start with 19 genuine flight records
test_df = sample_df.head(19).copy()

# Create deliberately invalid records
invalid_rows = []

# Test 1: Missing flight ID
row = sample_df.iloc[0].copy()
row["id"] = None
invalid_rows.append(row)

# Test 2: Missing source year
row = sample_df.iloc[1].copy()
row["source_year"] = None
invalid_rows.append(row)

# Test 3: Invalid source year
row = sample_df.iloc[2].copy()
row["source_year"] = 2020
invalid_rows.append(row)

# Test 4: Invalid source year
row = sample_df.iloc[3].copy()
row["source_year"] = 'test'
invalid_rows.append(row)

# Test 5: Missing source month
row = sample_df.iloc[4].copy()
row["source_month"] = None
invalid_rows.append(row)

# Test 6: Invalid source month
row = sample_df.iloc[5].copy()
row["source_month"] = 'test'
invalid_rows.append(row)

# Test 7: Source month and source year mismatch
row = sample_df.iloc[6].copy()
row["source_month"] = '202301'
row["source_year"] = '2024'
invalid_rows.append(row)

# Test 8: Missing flight date
row = sample_df.iloc[7].copy()
row["dof"] = None
invalid_rows.append(row)

# Test 9: Invalid flight date
row = sample_df.iloc[8].copy()
row["dof"] = 'test'
invalid_rows.append(row)

# Test 10: Flight Date and source year mismatch
row = sample_df.iloc[9].copy()
row["dof"] = '2023-01-01'
row["source_month"] = '202401'
row['source_year'] = 2024
invalid_rows.append(row)

# Test 11: Flight Date and source month mismatch
row = sample_df.iloc[10].copy()
row["source_year"] = 2023
row["source_month"] = "202309"
row["dof"] = "2023-08-20"
invalid_rows.append(row)

# Test 12: Missing expected duration min
row = sample_df.iloc[11].copy()
row["expected_duration_min"] = None
invalid_rows.append(row)

# Test 13: Invalid expected duration min
row = sample_df.iloc[12].copy()
row["expected_duration_min"] = 'test'
invalid_rows.append(row)

# Test 14: Invalid expected duration min
row = sample_df.iloc[13].copy()
row["expected_duration_min"] = -1
invalid_rows.append(row)

# Test 15: Invalid target value
row = sample_df.iloc[14].copy()
row["delayed_15min"] = 5
invalid_rows.append(row)

# Test 16: Target value delay min mismatch
row = sample_df.iloc[15].copy()
row["delayed_15min"] = 1
row["delay_min"] = 15
invalid_rows.append(row)

# Test 17: Target value delay min mismatch
row = sample_df.iloc[16].copy()
row["delayed_15min"] = 1
row["delay_min"] = 10
invalid_rows.append(row)

# Test 18: Target value delay min mismatch
row = sample_df.iloc[17].copy()
row["delayed_15min"] = 0
row["delay_min"] = 16
invalid_rows.append(row)

# Combine valid and invalid records
invalid_df = pd.DataFrame(invalid_rows)

test_df = pd.concat(
    [test_df, invalid_df],
    ignore_index=True
)

# Convert object fields to strings so invalid values can be tested
test_df["dof"] = test_df["dof"].astype("string")
test_df["source_year"] = test_df["source_year"].astype("string")
test_df["expected_duration_min"] = test_df["expected_duration_min"].astype("string")

# Preserve large flight identifiers without losing precision
test_df["id"] = pd.array(test_df["id"], dtype="UInt64")

# Save the validation test dataset
INVALID_SAMPLE = WORK_DIR / "flights_validation_test.parquet"
test_df.to_parquet(INVALID_SAMPLE, index=False)

print("Total test records:", len(test_df))
display(test_df[["id", "source_month", "delayed_15min", "dof"]])

## 9. Execute pipeline against the new test dataset containing invalid records

In [ ]:
# Separate output prefix to preserve previous results
VALIDATION_PREFIX = str(
    OUTPUT_DIR / "flights_validation_test"
)

with beam.Pipeline(runner="DirectRunner") as pipeline:

    fp.build(
        pipeline,
        input_pattern=str(INVALID_SAMPLE),
        output_prefix=VALIDATION_PREFIX
    )

print("Validation test pipeline completed.")

## 10. Inspect the quarantined records

In [ ]:
validation_good_files = sorted(
    OUTPUT_DIR.glob("flights_validation_test_good*.jsonl")
)

validation_bad_files = sorted(
    OUTPUT_DIR.glob("flights_validation_test_bad*.jsonl")
)

validation_good = read_jsonl_files(validation_good_files)
validation_bad = read_jsonl_files(validation_bad_files)

print("Input records:", len(test_df))
print("Accepted records:", len(validation_good))
print("Rejected records:", len(validation_bad))

if not validation_bad.empty:
    display(validation_bad[["reason", "id"]])


## 11. Reconcile and verify

In [ ]:
input_count = len(test_df)
good_count = len(validation_good)
bad_count = len(validation_bad)

assert input_count == good_count + bad_count, (
    "Reconciliation failed, total count incorrect"
)

assert 19 == good_count, ("Reconciliation failed, good record count incorrect expected 19 but was " + str(good_count))

assert 18 == bad_count, ("Reconciliation failed, bad record count incorrect expected 18 but was " + str(bad_count))

print("Reconciliation passed.")

expected_reasons = {
    "missing_source_year",
    "missing_id",
    "invalid_expected_duration",
    "missing_flight_date",
    "invalid_flight_date",
    "flight_year_mismatch",
    "flight_month_mismatch",
    "missing_source_month",
    "invalid_source_month",
    "invalid_source_year",
    "missing_expected_duration",
    "target_delay_mismatch",
    "invalid_target",
}

actual_reasons = set(
    validation_bad["reason"].tolist()
) if not validation_bad.empty else set()

print("Expected reasons:", expected_reasons)
print("Actual reasons:", actual_reasons)

assert expected_reasons == actual_reasons, (
    f"Unexpected rejection reasons. "
    f"Missing: {expected_reasons - actual_reasons}. "
    f"Unexpected: {actual_reasons - expected_reasons}"
)

print("All expected rejection reasons verified.")

## 12. Create combined validation and duplicate test dataset

This test extends the existing 37-record validation dataset by
adding one duplicate of a valid flight.

Expected results:
- 18 accepted records
- 18 rejected for invalid data
- 2 rejected for duplicate flight IDs
- 38 total input records

In [ ]:

# Start with our existing 37-record validation dataset
combined_df = test_df.copy()

# Duplicate the first genuine flight record
duplicate_row = test_df.iloc[[0]].copy()

# Add it to the dataset
combined_df = pd.concat(
    [combined_df, duplicate_row],
    ignore_index=True
)

# Preserve the original large flight ID values
combined_df["id"] = pd.array(
    combined_df["id"], dtype="UInt64"
)

# Save the combined test dataset
COMBINED_SAMPLE = WORK_DIR / "flights_combined_test.parquet"

combined_df.to_parquet(
    COMBINED_SAMPLE,
    index=False
)

print("Total input records:", len(combined_df))
print("Duplicated flight ID:", duplicate_row.iloc[0]["id"])


## 13. Execute Combined Validation and Duplicate Test

In [ ]:

COMBINED_PREFIX = str(
    OUTPUT_DIR / "flights_combined_test"
)

# Remove previous output files for this specific test
for old_file in OUTPUT_DIR.glob(
    "flights_combined_test_*.jsonl"
):
    old_file.unlink()

with beam.Pipeline(runner="DirectRunner") as pipeline:

    fp.build(
        pipeline,
        input_pattern=str(COMBINED_SAMPLE),
        output_prefix=COMBINED_PREFIX
    )

print("Combined pipeline test completed.")


## 14. Inspect and reconcile the combined results

In [ ]:

# Read accepted and quarantined records
combined_good_files = sorted(
    OUTPUT_DIR.glob("flights_combined_test_good*.jsonl")
)

combined_bad_files = sorted(
    OUTPUT_DIR.glob("flights_combined_test_bad*.jsonl")
)

combined_good = read_jsonl_files(combined_good_files)
combined_bad = read_jsonl_files(combined_bad_files)

# Count records rejected for duplicate IDs
duplicate_count = (
    combined_bad["reason"] == "duplicate_flight_id"
).sum()

# Count records rejected for other validation reasons
invalid_count = len(combined_bad) - duplicate_count

good_count = len(combined_good)
input_count = len(combined_df)

print("Input records:     ", input_count)
print("Accepted records:  ", good_count)
print("Invalid records:   ", invalid_count)
print("Duplicate records: ", duplicate_count)

# Reconcile input and output counts
assert input_count == good_count + invalid_count + duplicate_count

# Check expected results
assert input_count == 38
assert good_count == 18
assert invalid_count == 18
assert duplicate_count == 2

# Verify both copies of the duplicated flight were rejected
duplicate_id = str(duplicate_row.iloc[0]["id"])

duplicate_rejections = combined_bad[
    combined_bad["reason"] == "duplicate_flight_id"
]

assert (
    duplicate_rejections["id"].astype(str) == duplicate_id
).all()

print("All combined validation checks passed.")

display(combined_bad["reason"].value_counts())


## 15. Validate the Full Training Dataset

In [ ]:

# Full training dataset already downloaded in Section 5
FULL_TRAIN_PREFIX = str(
    OUTPUT_DIR / "flights_full_train"
)

# Remove previous outputs for this test only
for old_file in OUTPUT_DIR.glob("flights_full_train_*.jsonl"):
    old_file.unlink()

# Count input records
train_input_count = len(pd.read_parquet(
    LOCAL_TRAIN, columns=["id"]
))

print("Training input records:", train_input_count)

# Run the complete Apache Beam pipeline
with beam.Pipeline(runner="DirectRunner") as pipeline:
    fp.build(
        pipeline,
        input_pattern=str(LOCAL_TRAIN),
        output_prefix=FULL_TRAIN_PREFIX
    )

print("Full training pipeline completed.")


In [ ]:
# Inspect the results


# Find the output files
train_good_files = sorted(
    OUTPUT_DIR.glob("flights_full_train_good*.jsonl")
)

train_bad_files = sorted(
    OUTPUT_DIR.glob("flights_full_train_bad*.jsonl")
)

# Ensure output files exist
assert train_good_files, "No accepted-output files found"
assert train_bad_files, "No quarantine-output files found"

# Reuse the function defined in Section 6
train_good = read_jsonl_files(train_good_files)
train_bad = read_jsonl_files(train_bad_files)

# Calculate totals
train_good_count = len(train_good)
train_bad_count = len(train_bad)

print("Input records:      ", train_input_count)
print("Accepted records:   ", train_good_count)
print("Quarantined records:", train_bad_count)

# Verify reconciliation
assert (
    train_input_count == train_good_count + train_bad_count
), "Training reconciliation failed"

# Verify accepted flight IDs are unique
assert not train_good["id"].astype(str).duplicated().any(), (
    "Duplicate IDs found in accepted records"
)

# Verify rejected records have a reason
if not train_bad.empty:
    assert train_bad["reason"].notna().all()
    assert train_bad["reason"].astype(str).str.strip().ne("").all()

print("All full training data-quality checks passed.")

# Summarise quarantine reasons
if not train_bad.empty:
    display(
        train_bad["reason"]
        .value_counts()
        .rename_axis("reason")
        .reset_index(name="count")
    )
else:
    print("No quarantined records.")


## 16. Validate Development and Test Datasets

In [ ]:

# Download the dev split created by Notebook 03
SOURCE_OBJECT = f"{SPLIT_PREFIX}/flights_dev.parquet"

blob = bucket.blob(SOURCE_OBJECT)

if not blob.exists():
    raise FileNotFoundError(
        f"Training file not found: gs://{BUCKET_NAME}/{SOURCE_OBJECT}"
    )

blob.download_to_filename(str(LOCAL_DEV))


# Download the test split created by Notebook 03
SOURCE_OBJECT = f"{SPLIT_PREFIX}/flights_test.parquet"

blob = bucket.blob(SOURCE_OBJECT)

if not blob.exists():
    raise FileNotFoundError(
        f"Training file not found: gs://{BUCKET_NAME}/{SOURCE_OBJECT}"
    )

blob.download_to_filename(str(LOCAL_TEST))


# Process the remaining two datasets
remaining_splits = {
    "dev": LOCAL_DEV,
    "test": LOCAL_TEST
}

for split_name, input_path in remaining_splits.items():

    output_prefix = str(
        OUTPUT_DIR / f"flights_full_{split_name}"
    )

    # Remove previous output files
    for old_file in OUTPUT_DIR.glob(
        f"flights_full_{split_name}_*.jsonl"
    ):
        old_file.unlink()

    # Count source records
    input_count = len(
        pd.read_parquet(input_path, columns=["id"])
    )

    # Run the Apache Beam pipeline
    with beam.Pipeline(runner="DirectRunner") as pipeline:
        fp.build(
            pipeline,
            input_pattern=str(input_path),
            output_prefix=output_prefix
        )

    # Read the pipeline outputs
    good_files = sorted(
        OUTPUT_DIR.glob(f"flights_full_{split_name}_good*.jsonl")
    )
    bad_files = sorted(
        OUTPUT_DIR.glob(f"flights_full_{split_name}_bad*.jsonl")
    )

    assert good_files, f"No good output for {split_name}"

    good_df = read_jsonl_files(good_files)
    bad_df = read_jsonl_files(bad_files)

    # Check reconciliation
    assert len(good_df) + len(bad_df) == input_count

    # Check uniqueness
    assert not good_df["id"].astype(str).duplicated().any()

    print(
        f"{split_name.upper()}: "
        f"{input_count} input, "
        f"{len(good_df)} accepted, "
        f"{len(bad_df)} quarantined"
    )

print("Development and test validation completed.")


## 17. Check for Flight ID Overlap Between Dataset Splits

Verify that no flight ID appears in more than one dataset.

This prevents the same flight record from appearing in both
training and evaluation data.

In [ ]:

# Read flight IDs from each dataset
train_ids = set(
    pd.read_parquet(LOCAL_TRAIN, columns=["id"])["id"].astype(str)
)

dev_ids = set(
    pd.read_parquet(LOCAL_DEV, columns=["id"])["id"].astype(str)
)

test_ids = set(
    pd.read_parquet(LOCAL_TEST, columns=["id"])["id"].astype(str)
)

# Check overlaps between each pair of datasets
train_dev_overlap = train_ids.intersection(dev_ids)
train_test_overlap = train_ids.intersection(test_ids)
dev_test_overlap = dev_ids.intersection(test_ids)

print("Training vs Development:", len(train_dev_overlap))
print("Training vs Test:       ", len(train_test_overlap))
print("Development vs Test:    ", len(dev_test_overlap))

# Confirm no flight IDs are shared between datasets
assert len(train_dev_overlap) == 0, "Train/Dev overlap detected"
assert len(train_test_overlap) == 0, "Train/Test overlap detected"
assert len(dev_test_overlap) == 0, "Dev/Test overlap detected"

print("\nPASS: No flight ID overlap between datasets.")
